# Advanced Lab · VLM Structured Driving Conditions

Optional lab. VLM 的安全接口不是“让语言模型直接控制方向盘”，而是把图像/场景描述转成 schema-constrained、可验证、可拒答的 driving conditions，交给既有 BEV/planning/safety contract。

当前实验不下载大型 checkpoint；`requirements-frontier.txt` 只在你把 `predict_conditions` 替换为真实 Hugging Face processor/model 时安装。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import json
import numpy as np

SCHEMA = {"road_work": bool, "occluded_actor": bool, "cut_in_risk": float,
          "confidence": float, "evidence": list}

def validate_conditions(payload):
    required = set(SCHEMA)
    missing = required - set(payload)
    valid = not missing and 0.0 <= payload.get("cut_in_risk", -1) <= 1.0 and 0.0 <= payload.get("confidence", -1) <= 1.0
    return {"valid": valid, "missing": sorted(missing)}

observation = {
    "road_work": False, "occluded_actor": True, "cut_in_risk": 0.82,
    "confidence": 0.71, "evidence": ["left actor crosses lane boundary", "partial occlusion"],
}
print(validate_conditions(observation))
print(json.dumps(observation, indent=2))

练习：加入 abstain/unknown 状态，构造 schema-valid but semantically-wrong output，并把低 confidence 或 contradiction 送入 Chapter 09 safety gate。评估 precision/recall/coverage，而不是只展示一段漂亮文字。